# TinyStories-91M — original training run

This is the Colab notebook that produced the checkpoint described in the
[README](../README.md): a 91M-parameter decoder-only transformer trained from
scratch on TinyStories, reaching **1.170 validation loss (ppl 3.22)** in 49
minutes on an A100.

We put the same code, cleaned up and split into modules, in [`src/`](../src).

Cell order: setup → `config` → `model` → `dataset` → `train` → `generate` → run.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import random
import numpy as np
import torch

def set_seed(seed):
  random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)


In [ ]:

# config.py

from pathlib import Path

def get_config():
  return {
      # dataset
      "datasource": 'roneneldan/TinyStories',
      "cache_dir": "/content/drive/MyDrive/tinystories_cache",
      "num_rows": 2_110_000,
      "vocab_size": 8192,

      # model
      "d_model": 768,
      "N": 12,
      "h": 12,
      "max_seq": 512,
      "rope_max_seq": 1024,

      # training
      "num_epochs": 2,
      "batch_size": 128,
      "lr": 1e-3,
      "betas": (0.9, 0.95),
      "weight_decay": 0.1,
      "grad_clip": 1.0,
      "warmup_ratio": 0.03,
      "decay_ratio": 0.15,
      "label_smoothing": 0.0,
      "val_every_pct": 0.05,
      "seed": 0,
      "ckpt_every_pct": 0.25,
      "ckpt_path": "/content/drive/MyDrive/tinystories_cache/ckpt.pt",
  }

def tokenizer_path(cfg: dict) -> Path:
  d = Path(cfg["cache_dir"])
  d.mkdir(parents=True, exist_ok=True)
  dataset_name = cfg["datasource"].split("/")[-1]
  vocab_size = cfg["vocab_size"]
  num_rows = cfg["num_rows"]
  return d / f"{dataset_name}_{vocab_size}_{num_rows}_tokenizer.json"

In [ ]:
# model.py
import torch
import torch.nn as nn
import torch.nn.functional as F


def rotate_half(x):
  x1, x2 = x.chunk(2, dim=-1)
  return torch.cat((-x2, x1), dim=-1)

def apply_rope(q, k, cos, sin):
  cos, sin = cos[None, None], sin[None, None] #shape is (1, 1, seq_len, head_dim)
  in_dtype = q.dtype
  q, k = q.float(), k.float()
  q_rot = q * cos + rotate_half(q) * sin
  k_rot = k * cos + rotate_half(k) * sin
  return q_rot.to(in_dtype), k_rot.to(in_dtype)

class RoPE(torch.nn.Module):
  def __init__(self, head_dim, max_seq=512, base=10000.0):
    super().__init__()
    assert head_dim % 2 == 0, f"the head dimension {head_dim} should be even"
    inv = base ** (-torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim)
    freqs = torch.outer(torch.arange(max_seq, dtype=torch.float32), inv)
    emb = torch.cat((freqs, freqs), dim=-1)        # [max_seq, head_dim]
    self.register_buffer("cos", emb.cos(), persistent=False)
    self.register_buffer("sin", emb.sin(), persistent=False)

  def forward(self, seq_len, offset=0):
    s = slice(offset, offset + seq_len)
    return self.cos[s], self.sin[s]

class CausalMHA(nn.Module):
  def __init__(self, d_model: int, h: int):
    super().__init__()
    assert d_model % h == 0, f"{d_model} is not divisible by {h}"
    self.d_model= d_model
    self.h = h
    self.d_k = d_model // h
    self.w_q = nn.Linear(d_model, d_model, bias = False)
    self.w_k = nn.Linear(d_model, d_model, bias = False)
    self.w_v = nn.Linear(d_model, d_model, bias = False)
    self.w_o = nn.Linear(d_model, d_model, bias = False)

  def forward(self, x, cos, sin, cache = None):
    split_heads = lambda x: x.reshape(x.shape[0], x.shape[1], self.h, self.d_k).swapaxes(1, 2)
    # x is of shape (B, seq_len, d_model)
    q, k, v = split_heads(self.w_q(x)), split_heads(self.w_k(x)), split_heads(self.w_v(x))
    q, k = apply_rope(q, k, cos, sin)
    if cache is not None:
      if "k" in cache:
        k, v = torch.cat([cache["k"], k], 2), torch.cat([cache["v"], v], 2)
      cache["k"], cache["v"] = k, v
    out = F.scaled_dot_product_attention(q, k, v, None, 0.0, is_causal=q.shape[2] > 1)
    out = out.swapaxes(1, 2).reshape(out.shape[0], out.shape[2], self.d_model)
    return self.w_o(out)


class SwiGLU(nn.Module):
  def __init__(self, d_model):
    super().__init__()
    self.d_ffn = round((8*d_model) / (3*256)) * 256 # round to nearest multiple of 256
    self.w_g = nn.Linear(d_model, self.d_ffn, bias=False)
    self.w_v = nn.Linear(d_model, self.d_ffn, bias=False)
    self.w_o = nn.Linear(self.d_ffn, d_model, bias=False)

  def forward(self, x):
    x = F.silu(self.w_g(x)) * self.w_v(x)
    return self.w_o(x)


class DecoderBlock(nn.Module):
  def __init__(self, d_model, h):
    super().__init__()
    self.causal_mha = CausalMHA(d_model, h)
    self.swiglu = SwiGLU(d_model)
    self.rmsnorm1 = nn.RMSNorm(d_model)
    self.rmsnorm2 = nn.RMSNorm(d_model)

  def forward(self, x, cos, sin, cache=None):
    h = x + self.causal_mha(self.rmsnorm1(x), cos, sin, cache)
    return h + self.swiglu(self.rmsnorm2(h))


class LM(nn.Module):
  def __init__(self, d_model, vocab_size, h, N, max_seq, rope_max):
    super().__init__()
    self.d_model = d_model
    self.embed = nn.Embedding(vocab_size, d_model)
    self.blocks = nn.ModuleList([DecoderBlock(d_model, h) for _ in range(N)])
    self.rmsnorm = nn.RMSNorm(d_model)
    self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
    self.lm_head.weight = self.embed.weight
    self.rope = RoPE(d_model // h, rope_max)
    self._init_weights()

  def _init_weights(self):
    std = self.d_model ** -0.5
    scaled_std = std / (2 * len(self.blocks)) ** 0.5
    for name, module in self.named_modules():
      if isinstance(module, nn.Linear) and name.endswith("w_o"):
        nn.init.normal_(module.weight, mean=0.0, std=scaled_std)
      elif isinstance(module, (nn.Linear, nn.Embedding)):
        nn.init.normal_(module.weight, mean=0.0, std=std)

  def forward(self, x, past_len=0, caches=None):
    x = self.embed(x)
    cos, sin = self.rope(x.size(1), offset=past_len)

    for i, block in enumerate(self.blocks):
      x = block(x, cos, sin, None if caches is None else caches[i])
    return self.lm_head(self.rmsnorm(x))


In [ ]:
# dataset.py

from datasets import load_dataset

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.trainers import BpeTrainer
from tokenizers import decoders

import torch
from torch.utils.data import Dataset

import numpy as np

import gc

def fix_mojibake(s):
  if "â€" not in s and "Ã" not in s:
    return s
  for enc in ("cp1252", "latin-1"):
    try:
      return s.encode(enc).decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
      continue
  return s

def load_ds(cfg):
  train_ds = load_dataset(cfg["datasource"], split=f"train[:{cfg['num_rows']}]")
  val_ds = load_dataset(cfg["datasource"], split="validation")
  return train_ds, val_ds

def get_or_build_tokenizer(cfg, rows):
  path = tokenizer_path(cfg)
  if path.exists():
    return Tokenizer.from_file(str(path))

  tok = Tokenizer(BPE(unk_token="<unk>"))
  tok.pre_tokenizer = ByteLevel()
  tok.decoder = decoders.ByteLevel()


  trainer = BpeTrainer(
      vocab_size=cfg["vocab_size"],
      special_tokens=["<unk>", "<pad>", "<bos>", "<eos>"],
      initial_alphabet=ByteLevel.alphabet()
  )

  tok.train_from_iterator((fix_mojibake(row["text"]) for row in rows), trainer)
  tok.save(str(path))
  return tok


class TextDataset(Dataset):
  def __init__(self, raw_ds, tok, max_seq, batch=20_000):
    super().__init__()
    self.pad_id, self.bos_id, self.eos_id = map(tok.token_to_id, ("<pad>", "<bos>", "<eos>"))
    parts = []
    for s in range(0, len(raw_ds), batch):
      buf = []
      texts = [fix_mojibake(t) for t in raw_ds[s : s + batch]["text"]]
      for e in tok.encode_batch(texts):
        buf.append(self.bos_id); buf.extend(e.ids); buf.append(self.eos_id)
      parts.append(np.asarray(buf, dtype=np.int32))
    self.tokens = torch.from_numpy(np.concatenate(parts))
    del parts, buf; gc.collect()
    self.max_seq = max_seq
    self.num_samples = (len(self.tokens) - 1) // self.max_seq

  def __len__(self):
    return self.num_samples

  def __getitem__(self, idx):
    i = idx * self.max_seq
    return {"input_ids": self.tokens[i : i + self.max_seq].long(),
            "labels":    self.tokens[i + 1 : i + self.max_seq + 1].long()}


def _ds_from_cache(tok, tokens, max_seq, num_samples):
  ds = TextDataset.__new__(TextDataset)
  ds.pad_id, ds.bos_id, ds.eos_id = map(tok.token_to_id, ("<pad>", "<bos>", "<eos>"))
  ds.tokens, ds.max_seq, ds.num_samples = tokens, max_seq, num_samples
  return ds


In [ ]:

# train.py

import hashlib, time, pickle, math, gc
from tqdm import tqdm
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import LinearLR, ConstantLR, SequentialLR

import sys

_DS_MEMO = {}

def _ds_key(cfg):
  return "|".join(str(cfg[k]) for k in ["datasource", "num_rows", "vocab_size", "max_seq"])

def _ds_cache_file(cfg):
  h = hashlib.md5(_ds_key(cfg).encode()).hexdigest()[:12]
  d = Path(cfg["cache_dir"])
  d.mkdir(parents = True, exist_ok=True)
  return d / f"ds_{h}.pkl"

def get_ds(cfg):
  key, path, tok_path = _ds_key(cfg), _ds_cache_file(cfg), tokenizer_path(cfg)
  if key in _DS_MEMO:
    train_ds, val_ds, tok = _DS_MEMO[key]
    print("dataset: memory cache hit")

  elif path.exists() and tok_path.exists():
    print(f"dataset: loading {path}")
    t0 = time.time()
    with open(path, "rb") as f:
      cache = pickle.load(f)
    tok = Tokenizer.from_file(str(tok_path))
    train_ds = _ds_from_cache(tok, *cache["train"])
    val_ds = _ds_from_cache(tok, *cache["val"])
    _DS_MEMO[key] = (train_ds, val_ds, tok)
    print(f"dataset: loaded in {time.time()-t0:.0f}s")
  else:
    print("dataset: cold build")
    train_rows, val_rows = load_ds(cfg)
    tok = get_or_build_tokenizer(cfg, train_rows)
    train_ds = TextDataset(train_rows, tok, cfg["max_seq"])
    val_ds = TextDataset(val_rows, tok, cfg["max_seq"])
    del train_rows, val_rows ; gc.collect()

    tmp = path.with_suffix(".tmp")
    with open(tmp, "wb") as f:
      pickle.dump({"train": (train_ds.tokens, train_ds.max_seq, train_ds.num_samples),
                   "val": (val_ds.tokens, val_ds.max_seq, val_ds.num_samples)}, f, protocol=5)
    tmp.rename(path)
    _DS_MEMO[key] = (train_ds, val_ds, tok)
    print(f'dataset: cached to {path}')
  print(f"Vocab {tok.get_vocab_size()} | train {len(train_ds)} | val {len(val_ds)}")
  args = dict(batch_size = cfg['batch_size'], pin_memory=True, drop_last=True,
              persistent_workers=True, prefetch_factor=4)
  train_dl = DataLoader(train_ds, shuffle=True, num_workers=8, **args)
  val_dl = DataLoader(val_ds, num_workers=2, **args)
  return train_dl, val_dl, tok


@torch.inference_mode()
def run_validation_loss(model, val_dl, eval_loss_fn, vocab_size, device):
  model.eval()
  total_loss = 0.0
  for batch in val_dl:
    input_ids = batch["input_ids"].to(device, non_blocking=True)
    labels = batch["labels"].to(device, non_blocking=True)

    with torch.autocast("cuda", dtype=torch.bfloat16):
      logits = model(input_ids)
      loss = eval_loss_fn(logits.view(-1, vocab_size), labels.view(-1))
    total_loss += loss.item()

  return total_loss / len(val_dl)

def train_model(cfg):
  assert torch.cuda.is_available(), "this codebase is CUDA-only"
  torch.backends.cudnn.benchmark = True
  device = torch.device("cuda")
  torch.set_float32_matmul_precision("high")
  train_dl, val_dl, tok = get_ds(cfg)
  set_seed(cfg["seed"])

  model = LM(cfg["d_model"], tok.get_vocab_size(), cfg["h"], cfg["N"],
             cfg["max_seq"], cfg["rope_max_seq"]).to(device)
  raw_model = model
  print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

  total_steps  = len(train_dl) * cfg["num_epochs"]
  warmup_steps = max(100, int(cfg["warmup_ratio"] * total_steps))
  decay_steps  = int(cfg["decay_ratio"] * total_steps)
  stable_steps = total_steps - warmup_steps - decay_steps

  opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], betas=cfg["betas"],
                          weight_decay=cfg["weight_decay"], fused=True)
  warmup = LinearLR(opt, start_factor=0.01, end_factor=1.0, total_iters=warmup_steps)
  stable = ConstantLR(opt, factor=1.0, total_iters=stable_steps)
  decay  = LinearLR(opt, start_factor=1.0, end_factor=0.0, total_iters=decay_steps)
  scheduler = SequentialLR(opt, schedulers=[warmup, stable, decay],
                           milestones=[warmup_steps, warmup_steps + stable_steps])

  val_every = max(1, round(cfg["val_every_pct"] * total_steps))
  ckpt_every = max(1, round(cfg["ckpt_every_pct"] * total_steps))
  print(f"steps {total_steps:,} | warmup {warmup_steps:,} | stable {stable_steps:,} "
        f"| decay {decay_steps:,} | val every {val_every:,}")

  step = 0
  model = torch.compile(model)

  loss_fn = nn.CrossEntropyLoss(label_smoothing=cfg["label_smoothing"])
  vocab = tok.get_vocab_size()
  val_hist = []

  for epoch in range(cfg["num_epochs"]):
    model.train()
    opt.zero_grad(set_to_none=True)
    it = tqdm(train_dl, desc=f"Epoch {epoch:02d}", file=sys.stdout, dynamic_ncols=True)
    ema, gn, seen = None, 0.0, 0

    for batch in it:
      input_ids = batch["input_ids"].to(device, non_blocking=True)
      labels    = batch["labels"].to(device, non_blocking=True)

      with torch.autocast("cuda", dtype=torch.bfloat16):
        logits = model(input_ids)
        loss = loss_fn(logits.view(-1, vocab), labels.view(-1))
      loss.backward()

      gn = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=cfg["grad_clip"])
      opt.step()
      opt.zero_grad(set_to_none=True)
      scheduler.step()
      step += 1

      cur = loss.item()
      ema = cur if ema is None else 0.98 * ema + 0.02 * cur
      it.set_postfix(avg=f"{ema:6.3f}", gn=f"{gn:5.2f}",
                     lr=f"{opt.param_groups[0]['lr']:.2e}")
      seen += labels.size(0)

      if step % val_every == 0:
        vl = run_validation_loss(raw_model, val_dl, loss_fn, vocab, device)
        val_hist.append((step, vl))
        it.write(f"[{100*step/total_steps:5.1f}% | step {step:,} | {seen:,} seqs] "
                 f"val loss {vl:6.3f} | val ppl {math.exp(vl):8.2f}")
        if step % ckpt_every == 0:
          torch.save({"model": raw_model.state_dict(), "cfg": cfg,
                      "step": step, "val_hist": val_hist}, cfg["ckpt_path"])
        model.train()

    it.close()
    val_loss = run_validation_loss(raw_model, val_dl, loss_fn, vocab, device)
    print(f"Epoch {epoch:02d} | train {ema:6.3f} | val loss {val_loss:6.3f}"
          f" | val ppl {math.exp(val_loss):8.2f}")

  return raw_model, tok, val_hist

In [ ]:
import torch, torch.nn.functional as F

@torch.inference_mode()
def generate(model, tok, prompt="", max_new=250, temp=0.8, top_k=50, device="cuda"):
  model.eval()
  ids = [tok.token_to_id("<bos>")] + tok.encode(prompt).ids
  eos = tok.token_to_id("<eos>")
  x = torch.tensor([ids], dtype=torch.long, device=device)
  caches, past, out = [{} for _ in model.blocks], 0, []
  for _ in range(max_new):
    with torch.autocast("cuda", dtype=torch.bfloat16):
      logits = model(x, past_len=past, caches=caches)[:, -1].float()
    past += x.shape[1]
    logits = logits / temp
    kth = torch.topk(logits, top_k).values[:, -1:]
    probs = F.softmax(logits.masked_fill(logits < kth, float("-inf")), dim=-1)
    nxt = torch.multinomial(probs, 1)
    if nxt.item() == eos:
      break
    out.append(nxt.item())
    x = nxt                      # only the new token goes in next round
  return prompt + tok.decode(out)

def chat(model, tok, **kw):
  while True:
    p = input("prompt> ").strip()
    if p in {"", "quit", "exit"}:
      break
    print(generate(model, tok, p, **kw), "\n")



In [ ]:
cfg = get_config()
model, tok, val_hist = train_model(cfg)
torch.save({"model": model.state_dict(), "cfg": cfg, "val_hist": val_hist},
           "/content/drive/MyDrive/tinystories_cache/lm_768x12.pt")
chat(model, tok)

dataset: cold build


dataset: cached to /content/drive/MyDrive/tinystories_cache/ds_d888f20f21dd.pkl
Vocab 8192 | train 911331 | val 9204
Parameters: 91,245,312
steps 14,238 | warmup 427 | stable 11,676 | decay 2,135 | val every 712
[  5.0% | step 712 | 91,136 seqs] val loss  1.791 | val ppl     6.00
[ 10.0% | step 1,424 | 182,272 seqs] val loss  1.585 | val ppl     4.88
[ 15.0% | step 2,136 | 273,408 seqs] val loss  1.492 | val ppl     4.45
[ 20.0% | step 2,848 | 364,544 seqs] val loss  1.438 | val ppl     4.21
[ 25.0% | step 3,560 | 455,680 seqs] val loss  1.402 | val ppl     4.06
[ 30.0% | step 4,272 | 546,816 seqs] val loss  1.377 | val ppl     3.96
[ 35.0% | step 4,984 | 637,952 seqs] val loss  1.355 | val ppl     3.88
[ 40.0% | step 5,696 | 729,088 seqs] val loss  1.340 | val ppl     3.82
[ 45.0% | step 6,408 | 820,224 seqs] val loss  1.327 | val ppl     3.77
Epoch 00: 100%|██████████| 7119/7119 [24:29<00:00,  4.84it/s, avg=1.312, gn=0.12, lr=1.00e-03]
Epoch 00 | train  1.312 | val loss  1.314 | val 

In [ ]:
cfg = get_config()
model = LM(cfg["d_model"], 8192, cfg["h"], cfg["N"], cfg["max_seq"], cfg["rope_max_seq"]).cuda()
model.load_state_dict(torch.load("/content/drive/MyDrive/tinystories_cache/lm_768x12.pt")["model"])

tok = get_or_build_tokenizer(cfg, _)
chat(model, tok)

prompt> Once upon a time, camil, a little boy
Once upon a time, camil, a little boy named Timmy went on a walk with his mom. They walked and walked until they found a big, brown rock. Timmy said, "Wow, Mommy! That rock is big and brown!" His mom said, "Yes, it is. Do you want to sit on it?" Timmy said, "Yes, please!" and climbed up on top of the rock.

Suddenly, Timmy's knee started to hurt. He said, "Mommy, my knee hurts!" His mom said, "Let's sit down and see if we can fix it." They sat down on the rock and Timmy's mom put a band-aid on his knee. Timmy said, "Thanks, Mommy! That rock was a good idea." His mom said, "You're welcome, Timmy. Let's go home now." And they left the park, happy and content. 

prompt> Once upon a time, a little boy named Camil
Once upon a time, a little boy named Camil lived in a small house with his mom. Camil's mom was very strict and always said: "Camil, you must behave!"

One day, Camil wanted to play with his toys outside. As he went outside, he saw som